In [3]:
import shutil
import os
from tqdm import tqdm
import pandas as pd
from sklearn.model_selection import train_test_split

train_dir = '/kaggle/input/competitions/amia-public-challenge-2026/train/train'

train_csv_path= "/kaggle/input/competitions/amia-public-challenge-2026/train.csv"
image_size_df= pd.read_csv("/kaggle/input/competitions/amia-public-challenge-2026/img_size.csv")

RANDOM_SEED= 40
df= pd.read_csv(train_csv_path)

image_ids = df["image_id"].unique()
train_ids, val_ids = train_test_split(
    image_ids,
    test_size=0.2,
    random_state=RANDOM_SEED
)



yolo_dir = f"yolo_dataset"

def prep_yolo_split(image_ids, split):
    img_dir = os.path.join(yolo_dir, split, 'images')
    lbl_dir = os.path.join(yolo_dir, split, 'labels')
    os.makedirs(img_dir, exist_ok=True); os.makedirs(lbl_dir, exist_ok=True)

    for img_name in tqdm(image_ids, desc=f"YOLO {split}"):
        src = os.path.join(train_dir, img_name + ".png")
        if os.path.exists(src): shutil.copy2(src, os.path.join(img_dir, img_name + ".png"))

        rows = df[df['image_id'] == img_name]
        image_size= image_size_df.loc[image_size_df['image_id'] == img_name]
        x, y= image_size['dim1'], image_size['dim0'] # from kaggle:the original dimensions of the files: dim0 is the height and dim1 is the width.
        original_x, original_y= x.values[0], y.values[0]
        x_scale= 1024/original_x
        y_scale= 1024/original_y
        lines = []
        for _, r in rows.iterrows():
            if r['class_id'] == 14: continue
            xc = ((r['x_min']*x_scale + r['x_max']*x_scale)/2) / 1024
            yc = ((r['y_min']*y_scale + r['y_max']*y_scale)/2) / 1024
            w = (r['x_max']*x_scale - r['x_min']*x_scale) / 1024
            h = (r['y_max']*y_scale - r['y_min']*y_scale) / 1024
            assert 0 <= xc <= 1
            assert 0 <= yc <= 1
            assert 0 <= w <= 1
            assert 0 <= h <= 1
            lines.append(f"{r['class_id']} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        with open(os.path.join(lbl_dir, img_name + ".txt"), 'w') as f:
            f.write('\n'.join(lines))

prep_yolo_split(train_ids, 'train')
prep_yolo_split(val_ids, 'val')




YOLO val: 100%|██████████| 1715/1715 [00:13<00:00, 128.18it/s]


In [4]:
import yaml
yaml_path = os.path.join(yolo_dir, 'dataset.yaml')
CLASS_NAMES = [
        "Aortic enlargement", "Atelectasis", "Calcification", "Cardiomegaly",
        "Consolidation", "ILD", "Infiltration", "Lung Opacity", "Nodule/Mass",
        "Other lesion", "Pleural effusion", "Pleural thickening", 
        "Pneumothorax", "Pulmonary fibrosis", "No finding"
    ]
with open(yaml_path, 'w') as f:
    yaml.dump({
        'path': yolo_dir, 'train': 'train/images', 'val': 'val/images',
        'names': {i: n for i, n in enumerate(CLASS_NAMES[:-1])}
    }, f, default_flow_style=False)



In [7]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 28.7 MB/s eta 0:00:0000:01


In [9]:
from ultralytics import YOLO
import torch

yolo_model = YOLO('yolov8m.pt')  # or yolov8x.pt for larger model

yolo_results = yolo_model.train(
    data=yaml_path,
    epochs=10,
    imgsz=1024,
    batch=4,
    device=0 if torch.cuda.is_available() else 'cpu',
    patience=5,
    save=True,
    project="kaggle/working/outputs/yolo_runs",
    name="chest_xray",
    exist_ok=True,
    mosaic=1.0,      # Enable mosaic augmentation for rare classes
    mixup=0.1,
    copy_paste=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5, translate=0.1, scale=0.1, shear=2,
    fliplr=0.5,
)

Ultralytics 8.4.66 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_dataset/dataset.yaml, degrees=5, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=chest_xray, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=5, perspe

In [15]:
from pathlib import Path

for p in Path("/kaggle/working").rglob("best.pt"):
    print(p)

/kaggle/working/runs/detect/kaggle/working/outputs/yolo_runs/chest_xray/weights/best.pt


In [17]:
# ── Prepare test images ────────────────────────────────────────────────────────
test_img_dir  = '/kaggle/input/competitions/amia-public-challenge-2026/test/test'
test_csv_path= "/kaggle/input/competitions/amia-public-challenge-2026/test.csv"

# Use unique image IDs to avoid duplicates
test_df= pd.read_csv(test_csv_path)
test_image_ids = test_df["image_id"].unique()

yolo_inference_model = YOLO(
    "/kaggle/working/runs/detect/kaggle/working/outputs/yolo_runs/chest_xray/weights/best.pt"
)

yolo_preds = {}   # image_id → list of (class_id, conf, x1, y1, x2, y2)

for img_id in tqdm(test_image_ids, desc="YOLO inference"):
    img_path = os.path.join(test_img_dir, f"{img_id}.png")
    result   = yolo_inference_model.predict(
        img_path,
        conf    = 0.05,
        iou     = 0.50,
        verbose = False,
    )[0]

    dets = []
    if result.boxes is not None and len(result.boxes):
        for box in result.boxes:
            cls  = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            dets.append((cls, conf, x1, y1, x2, y2))

    yolo_preds[img_id] = dets

print(f"\nYOLO predictions collected for {len(yolo_preds)} images ✓")
n_with_dets = sum(1 for v in yolo_preds.values() if len(v) > 0)
print(f"Images with at least one detection: {n_with_dets} / {len(yolo_preds)}")

YOLO inference: 100%|██████████| 6427/6427 [08:49<00:00, 12.13it/s]


YOLO predictions collected for 6427 images ✓
Images with at least one detection: 1638 / 6427


In [35]:
def preds_to_prediction_string(dets, img_id):
    """Convert detection list to the competition PredictionString format."""
    #dets = scale_boxes_to_original(dets, img_id)

    if not dets:
        return "14 1.0 0 0 1 1"

    parts = []
    for (cls, conf, x1, y1, x2, y2) in dets:
        parts.append(f"{int(cls)} {conf:.4f} {int(x1)} {int(y1)} {int(x2)} {int(y2)}")
    return " ".join(parts)
    
"""Build and save a submission CSV."""
rows = []
for img_id in test_image_ids:
    dets = yolo_preds.get(img_id, [])
    rows.append({"image_id": img_id, "PredictionString": preds_to_prediction_string(dets, img_id)})
    sub_df = pd.DataFrame(rows)
    path   = "/kaggle/working/submission.csv"
    sub_df.to_csv(path, index=False)
    print(f"  Saved [YOLOv8m] → submission  ({len(sub_df)} rows)")

  Saved [YOLOv8m] → submission  (1 rows)
  Saved [YOLOv8m] → submission  (2 rows)
  Saved [YOLOv8m] → submission  (3 rows)
  Saved [YOLOv8m] → submission  (4 rows)
  Saved [YOLOv8m] → submission  (5 rows)
  Saved [YOLOv8m] → submission  (6 rows)
  Saved [YOLOv8m] → submission  (7 rows)
  Saved [YOLOv8m] → submission  (8 rows)
  Saved [YOLOv8m] → submission  (9 rows)
  Saved [YOLOv8m] → submission  (10 rows)
  Saved [YOLOv8m] → submission  (11 rows)
  Saved [YOLOv8m] → submission  (12 rows)
  Saved [YOLOv8m] → submission  (13 rows)
  Saved [YOLOv8m] → submission  (14 rows)
  Saved [YOLOv8m] → submission  (15 rows)
  Saved [YOLOv8m] → submission  (16 rows)
  Saved [YOLOv8m] → submission  (17 rows)
  Saved [YOLOv8m] → submission  (18 rows)
  Saved [YOLOv8m] → submission  (19 rows)
  Saved [YOLOv8m] → submission  (20 rows)
  Saved [YOLOv8m] → submission  (21 rows)
  Saved [YOLOv8m] → submission  (22 rows)
  Saved [YOLOv8m] → submission  (23 rows)
  Saved [YOLOv8m] → submission  (24 rows)
 

In [3]:
import pandas as pd
sub = pd.read_csv("../submissions/02_submission.csv")

print(sub.shape)
print(sub["image_id"].nunique())

dups = sub[sub["image_id"].duplicated(keep=False)]
print(dups.head(20))

(6427, 2)
4
   image_id                                   PredictionString
0    image0                                     14 1.0 0 0 1 1
1    image1                                     14 1.0 0 0 1 1
2    image2                                     14 1.0 0 0 1 1
3    image3                   8 0.5815 699.7 351.3 859.8 541.2
4    image0                                     14 1.0 0 0 1 1
5    image1  3 0.7746 404.2 546.2 905.5 758.0 0 0.6163 544....
6    image2                                     14 1.0 0 0 1 1
7    image3                   0 0.3874 543.6 231.9 655.8 340.4
8    image0                                     14 1.0 0 0 1 1
9    image1                                     14 1.0 0 0 1 1
10   image2                                     14 1.0 0 0 1 1
11   image3                                     14 1.0 0 0 1 1
12   image0                                     14 1.0 0 0 1 1
13   image1                                     14 1.0 0 0 1 1
14   image2                                